# Model Call Plumbing

Level 1, Python Fundamentals in GenAI Context.

**Problems in this notebook**

- Problem 7: Conversation Memory Trimming
- Problem 8: Merge Streaming Partial Responses
- Problem 10: Calculate Token Cost per Customer

**How to use this notebook**

1. Read the problem and say your plan aloud before you write any code.
2. Paste the AI prompt given (or write your own narrow one) to fill the starter. Read the AI's code before you run it.
3. Run the checks. Open the reference solution only after you have tried.

The ideas behind these problems are in `04_model_call_plumbing_THEORY.md`.

In [ ]:
def run_checks(fn, cases):
    """cases: list of (args_tuple, expected). Prints a short report, never raises."""
    passed = 0
    for i, (args, expected) in enumerate(cases, 1):
        try:
            got = fn(*args)
        except NotImplementedError:
            print("Not implemented yet. Fill in the starter first.")
            return False
        except Exception as e:
            print(f"Case {i}: raised {type(e).__name__}: {e}")
            continue
        if got == expected:
            passed += 1
        else:
            print(f"Case {i}: expected {expected!r}, got {got!r}")
    print(f"{passed}/{len(cases)} checks passed")
    return passed == len(cases)

## Problem 7: Conversation Memory Trimming

**The situation**

A support chatbot sends the chat history to the model on every turn. The model has a token budget (the most text it will accept, counted in tokens, which are word pieces), and long chats go over it. You need to keep the system message plus as much recent history as fits.

**Your task**

Write `trim_conversation(messages, budget)`. Always keep messages with role `system`. Then add the newest other messages, going backwards, until the next one would go over the budget. Return the kept messages in their original order.

**Example**

Messages with tokens `[5, 10]` (oldest first) and budget 10 -> keeps only the newest, 10-token message.

**Ask before coding**

- Are messages ordered oldest to newest?
- Must the system message always stay, even if it uses most of the budget?
- Should we stop at the first message that does not fit, or skip it and keep looking at older ones?

**Plan to say aloud**

1. Keep every system message and count its tokens.
2. Walk the other messages from newest to oldest.
3. Keep each one while the total stays within the budget.
4. Stop at the first one that does not fit, so the history has no gaps.
5. Return the kept messages in their original order.

### Prompt for your AI assistant

```text
Write a Python function `trim_conversation(messages: list[dict], budget: int) -> list[dict]`. Each message has 'role' and 'tokens', oldest first. Always keep messages whose role is 'system' and count their tokens. Then walk the other messages from newest (end of list) to oldest, keeping each one while the total stays at or under budget; stop at the first that does not fit. Return the kept messages in their original order. Show only the function.
```

**Check the AI's code for**

- Drops the system message when the budget is tight.
- Walks from oldest to newest, so it keeps old chatter and loses the user's latest question.
- Skips an oversized message and keeps going, which leaves a hole in the conversation.
- Returns the kept messages in reversed order.

In [ ]:
def trim_conversation(messages: list[dict], budget: int) -> list[dict]:
    """Keep system messages plus the newest messages that fit in the token budget.

    Rules:
    - Messages are oldest first. Each has "role" and "tokens".
    - Always keep role == "system" messages; their tokens count toward the budget.
    - Walk the rest from newest to oldest. Keep each while the total <= budget.
      Stop at the first message that does not fit.
    - Return kept messages in their original order.
    """
    raise NotImplementedError

In [ ]:
def m(i, role, tokens):
    return {"id": i, "role": role, "tokens": tokens}


chat = [m(1, "system", 3), m(2, "user", 4), m(3, "assistant", 4), m(4, "user", 4)]
cases = [
    (([m(1, "user", 5), m(2, "assistant", 10)], 10), [m(2, "assistant", 10)]),
    ((chat, 11), [m(1, "system", 3), m(3, "assistant", 4), m(4, "user", 4)]),  # system always stays
    ((chat, 3), [m(1, "system", 3)]),
    (([m(1, "user", 1), m(2, "user", 9), m(3, "user", 2)], 5), [m(3, "user", 2)]),  # stop at first misfit, no gaps
    (([m(1, "user", 1)], 0), []),
    (([], 100), []),
]
run_checks(trim_conversation, cases)

### Reference solution

Try first. Then compare.

In [ ]:
def trim_conversation_reference(messages: list[dict], budget: int) -> list[dict]:
    keep = {i for i, msg in enumerate(messages) if msg["role"] == "system"}
    used = sum(messages[i]["tokens"] for i in keep)
    for i in range(len(messages) - 1, -1, -1):  # newest to oldest
        if i in keep:
            continue
        if used + messages[i]["tokens"] > budget:
            break  # stop here so the kept history has no gaps
        keep.add(i)
        used += messages[i]["tokens"]
    return [msg for i, msg in enumerate(messages) if i in keep]


assert run_checks(trim_conversation_reference, cases)

**Explain it in 30 seconds**

> "The model has a token budget, so I trim the history every turn. System messages always stay, because they hold the rules. Then I walk backwards from the newest message and keep messages until the next one would not fit, and I stop there so the history has no holes. I return what I kept in the original order."

**Edge cases to mention**

- The newest message alone is bigger than the budget: only the system message is kept.
- System messages that by themselves go over the budget.
- A budget of zero.
- Token counts that are estimates rather than real tokenizer counts.

**If they ask "how would this work in production?"**

- Count tokens with the model's real tokenizer, not words or characters.
- Summarize the dropped older turns into one short message instead of losing them completely.
- Track how often trimming happens and how much is dropped, because it affects answer quality.

## Problem 8: Merge Streaming Partial Responses

**The situation**

A model API streams its answer in small chunks. Most chunks carry a piece of text in `delta`, but some carry only metadata, such as usage counts. The backend must stitch the text pieces into the final answer.

**Your task**

Write `merge_stream(chunks)`. Join every string `delta`, in order, and ignore chunks without one. Do not change the text (no trimming).

**Example**

`[{"delta": "Hello"}, {"delta": " "}, {"delta": "world"}, {"usage": 3}]` -> `"Hello world"`

**Ask before coding**

- Do chunks always arrive in order?
- Can the same chunk arrive twice?
- If the stream fails halfway, do we keep the partial answer?

**Plan to say aloud**

1. Start an empty list of text parts.
2. For each chunk, read `delta` safely with `.get`.
3. Keep it only if it is a string.
4. Join the parts once, at the end.

### Prompt for your AI assistant

```text
Write a Python function `merge_stream(chunks: list[dict]) -> str`. For each chunk in order, if chunk.get('delta') is a string, collect it. Ignore chunks with no delta, a None delta, or a non-string delta. Do not strip or change the text. Join the parts once at the end with ''.join. Show only the function.
```

**Check the AI's code for**

- Uses `chunk['delta']` and crashes on metadata-only chunks.
- Turns `None` into the text `'None'` by calling `str()` on it.
- Strips each delta, which deletes the spaces between words.
- Builds the string with `+=` in a loop instead of one join at the end.

In [ ]:
def merge_stream(chunks: list[dict]) -> str:
    """Join the text deltas of a streamed model response.

    Rules:
    - Keep chunk["delta"] only when it is a str. Ignore every other chunk.
    - Keep the original order. Do not strip or change the text.
    """
    raise NotImplementedError

In [ ]:
cases = [
    (([{"delta": "Hello"}, {"delta": " "}, {"delta": "world"}, {"usage": 3}],), "Hello world"),
    (([{}, {"delta": None}, {"delta": "ok"}],), "ok"),
    (([{"delta": 5}, {"delta": "A"}, {"type": "ping"}, {"delta": "I"}],), "AI"),
    (([{"delta": ""}, {"delta": " keep spaces "}],), " keep spaces "),  # spaces are real content
    (([],), ""),
]
run_checks(merge_stream, cases)

### Reference solution

Try first. Then compare.

In [ ]:
def merge_stream_reference(chunks: list[dict]) -> str:
    parts = [c["delta"] for c in chunks if isinstance(c.get("delta"), str)]
    return "".join(parts)


assert run_checks(merge_stream_reference, cases)

**Explain it in 30 seconds**

> "Streaming APIs send many small chunks, and some carry no text at all. I read each chunk's delta safely with `.get`, keep it only if it is a string, and join everything once at the end. I do not trim, because spaces are real content. Joining once is cleaner than adding to a string over and over in a loop."

**Edge cases to mention**

- Metadata-only or keep-alive chunks.
- A `None` or numeric delta.
- Deltas that are only spaces.
- An error chunk that arrives after some text.

**If they ask "how would this work in production?"**

- Handle the user leaving mid-answer: cancel the upstream call so you stop paying for tokens.
- Save partial output and mark it as incomplete if the stream fails.
- Measure time to first token and total time, and run safety filters before showing the text.

## Problem 10: Calculate Token Cost per Customer

**The situation**

An enterprise GenAI platform bills each customer for model usage. Every call is logged with the customer, the model, and the input and output token counts. Prices are per 1,000 tokens and differ by model, and input tokens cost a different amount from output tokens.

**Your task**

Write `cost_per_customer(records, prices)`. Return the total cost per customer, rounded to 6 decimal places. Treat missing token counts as 0. Raise `ValueError` for an unknown model or a negative token count, instead of guessing.

**Example**

1,000 input tokens at 0.01 and 500 output tokens at 0.03 (per 1,000) -> `{"acme": 0.025}`

**Ask before coding**

- Are prices per 1,000 tokens or per 1,000,000 tokens?
- Should an unknown model fail the whole run, or be set aside for review?
- Can prices change over time, so we need the price at the time of each call?

**Plan to say aloud**

1. Start an empty dict of totals.
2. For each record, look up the model's price, and raise if it is missing.
3. Read the token counts, defaulting to 0, and raise if one is negative.
4. Add the input cost and output cost to that customer's total.
5. Round each total at the end.

### Prompt for your AI assistant

```text
Write a Python function `cost_per_customer(records: list[dict], prices: dict[str, dict[str, float]]) -> dict[str, float]`. Each record has 'customer', 'model', and optional 'input_tokens' and 'output_tokens' (default 0). prices[model] has 'input' and 'output' prices per 1,000 tokens. Sum the cost per customer and round each final total to 6 decimals. Raise ValueError for a model missing from prices or a negative token count. Show only the function.
```

**Check the AI's code for**

- Uses `prices.get(model, {})` with a default of 0, so unknown models are billed at zero and nobody notices.
- Divides by 1,000,000 instead of 1,000, or applies the input price to output tokens.
- Rounds every record instead of the final total, so small errors pile up.
- Crashes with KeyError when a record has no `output_tokens`.

In [ ]:
def cost_per_customer(records: list[dict], prices: dict[str, dict[str, float]]) -> dict[str, float]:
    """Total model cost per customer.

    Rules:
    - Each record: "customer", "model", optional "input_tokens" / "output_tokens" (default 0).
    - prices[model] = {"input": price per 1K tokens, "output": price per 1K tokens}.
    - Raise ValueError for an unknown model or a negative token count.
    - Round each customer's final total to 6 decimal places.
    """
    raise NotImplementedError

In [ ]:
PRICES = {"m1": {"input": 0.01, "output": 0.03}, "m2": {"input": 0.002, "output": 0.004}}


def cost_or_error(fn):
    """Wrap fn so a ValueError shows up as the string "ValueError" in the checks."""
    def wrapped(*args):
        try:
            return fn(*args)
        except ValueError:
            return "ValueError"
    return wrapped


cases = [
    (([{"customer": "acme", "model": "m1", "input_tokens": 1000, "output_tokens": 500}], PRICES), {"acme": 0.025}),
    (([{"customer": "acme", "model": "m1", "input_tokens": 1000, "output_tokens": 1000},
       {"customer": "acme", "model": "m2", "input_tokens": 2000},
       {"customer": "globex", "model": "m1", "output_tokens": 1000}], PRICES),
     {"acme": 0.044, "globex": 0.03}),
    (([{"customer": "acme", "model": "m2"}], PRICES), {"acme": 0.0}),  # missing counts mean 0
    (([], PRICES), {}),
    (([{"customer": "acme", "model": "unknown", "input_tokens": 10}], PRICES), "ValueError"),  # never bill at zero
    (([{"customer": "acme", "model": "m1", "input_tokens": -5}], PRICES), "ValueError"),
]
run_checks(cost_or_error(cost_per_customer), cases)

### Reference solution

Try first. Then compare.

In [ ]:
def cost_per_customer_reference(records: list[dict], prices: dict[str, dict[str, float]]) -> dict[str, float]:
    totals: dict[str, float] = {}
    for r in records:
        price = prices.get(r["model"])
        if price is None:
            raise ValueError(f"no price for model {r['model']!r}")
        tokens_in = r.get("input_tokens", 0)
        tokens_out = r.get("output_tokens", 0)
        if tokens_in < 0 or tokens_out < 0:
            raise ValueError(f"negative token count for customer {r['customer']!r}")
        cost = tokens_in / 1000 * price["input"] + tokens_out / 1000 * price["output"]
        totals[r["customer"]] = totals.get(r["customer"], 0.0) + cost
    return {customer: round(total, 6) for customer, total in totals.items()}


assert run_checks(cost_or_error(cost_per_customer_reference), cases)

**Explain it in 30 seconds**

> "I group usage by customer and add the input and output cost, using each model's own price per 1,000 tokens. Missing token counts count as zero, but an unknown model or a negative count raises an error, because billing must never guess quietly. I round only the final totals. For real billing I would use Decimal instead of float."

**Edge cases to mention**

- A model with no price: fail loudly, never bill it at zero.
- Negative or missing token counts.
- Prices that changed partway through the month.
- Float rounding, such as `0.1 + 0.2` not being exactly `0.3`.

**If they ask "how would this work in production?"**

- Use `Decimal` for money, and store usage events as records that are never edited.
- Version prices by date, and charge each call at the price in force when it happened.
- Give customers a usage dashboard and budget alerts per model and per time window.

## Wrap-up

- These are the small pieces around every model call: what goes in, what comes out, and what it costs.
- Read optional fields safely, and decide on purpose what to keep, skip, or reject.
- Each one has a clear production upgrade: a real tokenizer, stream cancellation, Decimal billing.